In [1]:
import time
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import DoubleType
import pyspark.sql.functions as F

# Create Spark session
spark = SparkSession.builder \
    .appName("Heavy_Computation_Test") \
    .master("local[10]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Using {spark.sparkContext.defaultParallelism} cores\n")

# ===== EXAMPLE: Computationally Heavy Operation =====
print("=" * 70)
print("HEAVY COMPUTATION: Monte Carlo simulation for each row")
print("=" * 70)

# Computationally expensive function: Monte Carlo simulation
def monte_carlo_pricing(price, volatility, time_horizon, n_simulations=10000):
    """
    Simulate stock price paths using geometric Brownian motion.
    This is O(n_simulations) per row - computationally expensive!
    """
    dt = time_horizon / 252  # Daily steps
    drift = -0.5 * volatility ** 2 * dt
    shock = volatility * np.sqrt(dt)
    
    paths = np.zeros(n_simulations)
    for i in range(n_simulations):
        daily_returns = np.random.normal(drift, shock, 252)
        paths[i] = price * np.exp(np.sum(daily_returns))
    
    return np.mean(paths)

# Create dataset
n_rows = 5000  # Fewer rows but HEAVY computation per row
print(f"\nCreating {n_rows} rows with heavy Monte Carlo computation per row...")

data = {
    'stock_id': range(n_rows),
    'price': np.random.uniform(50, 200, n_rows),
    'volatility': np.random.uniform(0.15, 0.40, n_rows),
    'time_horizon': np.random.uniform(0.5, 2.0, n_rows)
}

df_pandas = pd.DataFrame(data)

# ===== PANDAS APPROACH (Sequential) =====
print("\n--- PANDAS (Sequential, single-core) ---")
start = time.time()

df_pandas['simulated_price'] = df_pandas.apply(
    lambda row: monte_carlo_pricing(
        row['price'], 
        row['volatility'], 
        row['time_horizon']
    ), 
    axis=1
)

pandas_time = time.time() - start
print(f"Pandas time: {pandas_time:.2f} seconds")
print(f"First 5 results:\n{df_pandas[['stock_id', 'price', 'simulated_price']].head()}")

# ===== PYSPARK APPROACH (Parallel) =====
print("\n--- PYSPARK (Parallel, 10-core) ---")

# Define pandas UDF for Spark (this runs in parallel!)
@pandas_udf(DoubleType())
def monte_carlo_udf(price: pd.Series, volatility: pd.Series, time_horizon: pd.Series) -> pd.Series:
    """
    Pandas UDF - Spark will distribute chunks of data to different cores
    """
    result = []
    for p, v, t in zip(price, volatility, time_horizon):
        result.append(monte_carlo_pricing(p, v, t))
    return pd.Series(result)

# Create Spark DataFrame
df_spark = spark.createDataFrame(df_pandas[['stock_id', 'price', 'volatility', 'time_horizon']])

start = time.time()

# Apply the UDF - Spark distributes this across cores
df_spark = df_spark.withColumn(
    'simulated_price',
    monte_carlo_udf(
        F.col('price'),
        F.col('volatility'),
        F.col('time_horizon')
    )
)

# Trigger computation
result_spark = df_spark.toPandas()

spark_time = time.time() - start
print(f"PySpark time: {spark_time:.2f} seconds")
print(f"First 5 results:\n{result_spark[['stock_id', 'price', 'simulated_price']].head()}")

# ===== COMPARISON =====
print("\n" + "=" * 70)
print("RESULTS:")
print("=" * 70)
print(f"Pandas (sequential):  {pandas_time:.2f} seconds")
print(f"PySpark (parallel):   {spark_time:.2f} seconds")

speedup = pandas_time / spark_time
print(f"\nSpeedup: {speedup:.2f}x faster with PySpark")
print(f"Efficiency: {speedup/10*100:.1f}% (ideal would be 10x on 10 cores)")

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/26 09:10:06 WARN Utils: Your hostname, alpamayo.local, resolves to a loopback address: 127.0.0.1; using 10.0.1.21 instead (on interface en0)
26/01/26 09:10:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/26 09:10:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/26 09:10:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.1
Using 10 cores

HEAVY COMPUTATION: Monte Carlo simulation for each row

Creating 5000 rows with heavy Monte Carlo computation per row...

--- PANDAS (Sequential, single-core) ---
Pandas time: 188.27 seconds
First 5 results:
   stock_id       price  simulated_price
0         0   92.376374        91.932256
1         1  191.043046       190.930150
2         2  197.474145       197.625516
3         3  172.939431       173.127665
4         4   55.760323        55.562284

--- PYSPARK (Parallel, 10-core) ---


PySpark time: 38.65 seconds
First 5 results:
   stock_id       price  simulated_price
0         0   92.376374        91.878377
1         1  191.043046       191.391069
2         2  197.474145       196.670198
3         3  172.939431       174.571436
4         4   55.760323        55.807070

RESULTS:
Pandas (sequential):  188.27 seconds
PySpark (parallel):   38.65 seconds

Speedup: 4.87x faster with PySpark
Efficiency: 48.7% (ideal would be 10x on 10 cores)
